In [ ]:
!uv pip install -U requests python-dotenv openai

PR_JSON_FILE = "_prs.json"  # original PR entries
PR_TOPICS_FILE = "_topics.json"  # LLM summarized
MODEL = "gpt-5.6-terra"

Using Python 3.13.5 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 21 packages in 124ms                                        
Checked 21 packages in 0.77ms


In [ ]:
# Get all merged PRs (title, date, author)
# Please set `GITHUB_TOKEN` to increase GitHub rate limit

import json
import os

import requests

url = "https://api.github.com/repos/materialsproject/pymatgen/pulls"
params = {"state": "all", "per_page": 100, "page": 1}

# Setup GitHub token (for higher API rate limit)
if not (token := os.environ.get("GITHUB_TOKEN")):
    raise RuntimeError("Please set your GitHub token in GITHUB_TOKEN env var.")
headers = {"Authorization": f"Bearer {token}"}

# Load previous cache if exists
if os.path.exists(PR_JSON_FILE):
    print("Cache found, no fetch.")
else:
    pr_dict = {}  # {pr_number: {title, created_at, author}}

    while True:
        response = requests.get(url, headers=headers, params=params, timeout=60)
        response.raise_for_status()
        prs = response.json()
        if not prs:
            break
        for pr in prs:
            # Only keep merged PRs (comment out to keep all)
            if pr["merged_at"] is None:
                continue

            pr_dict[str(pr["number"])] = {
                "title": pr["title"],
                "created_at": pr["created_at"],
                "author": pr["user"]["login"],
            }
        params["page"] += 1  # type: ignore[unsupported-operator]

    with open(PR_JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(pr_dict, f, ensure_ascii=False)

    print(f"Saved {len(pr_dict)} PRs to {PR_JSON_FILE}")

Cache found, no fetch.


In [ ]:
# Quickly clean up PRs (drop bot PRs and so on)
import json

with open(PR_JSON_FILE, encoding="utf-8") as f:
    prs = json.load(f)  # {number: {...}}

filtered = {}
uniq_dropped_authors = set()

for num, data in prs.items():
    author = (data.get("author") or "").strip()

    if author.lower().endswith("[bot]"):
        if author:
            uniq_dropped_authors.add(author)
        continue

    filtered[num] = data

# Save cleaned
with open(PR_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(filtered, f, indent=2, ensure_ascii=False)

print(f"Kept {len(filtered)} PRs. Saved to {PR_JSON_FILE}")
print(f"Dropped by author: (unique authors: {len(uniq_dropped_authors)})")
for a in sorted(uniq_dropped_authors):
    print("  •", a)

Kept 2530 PRs. Saved to _prs.json
Dropped by title: (unique titles: 0)
Dropped by author: (unique authors: 0)


In [ ]:
# Feed PR titles to the LLM year-by-year and merge JSON results.

import json
from collections import defaultdict

from openai import OpenAI

client = OpenAI()

INSTRUCTIONS = """You are an expert technical summarizer.

You will receive GitHub pull request titles for ONE YEAR.
Identify the 6-10 main technical topics for that year.

For each topic, provide a summary of under 10 words and total number of PRs
under this topic.

Rules:
- Avoid personal names and issue numbers.
- Focus on technical subject matter.
- Do not invent topics; fewer than 6 is acceptable if warranted.

Return ONLY valid JSON in exactly this structure (no extra text, no Markdown):

{
  "year": YYYY,
  "topics": ["summary_1 (count_1)", "summary_2 (count_2)", ...]
}
"""


def get_output_text(resp) -> str:
    # SDK helper: prefer .output_text if present, else fall back
    try:
        return resp.output_text
    except AttributeError:
        return resp.output[0].content[0].text


def summarize_year(year: int, titles: list[str]) -> list[dict]:
    # Build compact per-year payload
    bullets = "\n".join(f"- {t}" for t in titles)
    prompt = f"""{INSTRUCTIONS}

YEAR {year} PR titles:
{bullets}
""".strip()

    resp = client.responses.create(
        model=MODEL,
        input=prompt,
        # max_output_tokens=700,
    )
    raw = get_output_text(resp)
    data = json.loads(raw)  # expect strict JSON due to prompt

    return data["topics"]


# Load PRs
with open(PR_JSON_FILE, encoding="utf-8") as f:
    prs = json.load(f)

# Group titles by year
by_year: dict[int, list[str]] = defaultdict(list)
for data in prs.values():
    title = (data.get("title") or "").strip()
    if not title:
        continue
    when = (data.get("merged_at") or data.get("created_at") or "").strip()
    if len(when) < 4:
        continue
    year = int(when[:4])
    by_year[year].append(title)

# Deduplicate per-year
for y in list(by_year.keys()):
    by_year[y] = sorted(set(by_year[y]))

# Call the model year-by-year and merge
merged: dict[str, list[str]] = {}
for year in sorted(by_year):
    print(f"Working on {year=}")
    topics = summarize_year(year, by_year[year])
    print(f"{topics=}\n")
    merged[str(year)] = topics  # type: ignore[invalid-assignment]

with open(PR_TOPICS_FILE, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

Working on year=2013
topics=['Abinit integration, formats, and code migration (10)', 'Molecular matching and RMSD algorithms (4)', 'Phase diagram visualization and entry handling (4)', 'Substitution and functional-group chemistry tools (4)', 'GULP, Zeo++, and defect-analysis interfaces (3)', 'Unit-aware arrays and serialization support (3)', 'Crystal symmetry, point groups, and k-path fixes (3)', 'Testing, bug fixes, and maintenance updates (5)', 'API properties, tools, and installation documentation (4)']

Working on year=2014
topics=['Defect transformations and dilute-solution thermodynamics (23)', 'Python 3 compatibility, utilities, and dependencies (12)', 'VASP parsing, dielectric properties, and input generation (10)', 'Abinit workflow, serialization, and GW support (7)', 'Bond-valence oxidation states and ionic radii (6)', 'FEFF, Gaussian, and core-level I/O (5)', 'Surface energies and terminations (4)']

Working on year=2015
topics=['Defect and dilute-solute modeling workflows (

In [ ]:
# Clean up PR summaries (as summary was done year by year,
# make sure wording and so on is consistent)

import json

from openai import OpenAI

client = OpenAI()

INSTRUCTIONS = """You are an expert technical editor.

You will receive a YEAR and that year's pull-request TOPICS (already summarized).
Your task: normalize wording so style is consistent across years.

Rules:
- Max ~8 topics; fewer is fine.
- Each topic under ~10 words.
- Use consistent terms: VASP, FHI-aims, CP2K, LOBSTER, JDFTx, OPTIMADE, JSON, I/O, CI.
- Avoid personal names and issue numbers.
- Leave the PR counts at the end of each topic untouched.
- Deduplicate/merge overlaps (merge counts in this case).

Return ONLY valid JSON in this exact structure:
{ "year": YYYY, "topics": ["..."] }
"""


def normalize_year(year: int, topics: list[str]) -> list[str]:
    prompt = f"""{INSTRUCTIONS}

YEAR {year} topics:
{chr(10).join(f"- {t}" for t in topics)}
"""
    resp = client.responses.create(
        model=MODEL,
        input=prompt,
    )
    # pull text from response
    try:
        raw = resp.output_text
    except AttributeError:
        raw = resp.output[0].content[0].text
    data = json.loads(raw)
    return data["topics"]


with open(PR_TOPICS_FILE, encoding="utf-8") as f:
    per_year = json.load(f)

cleaned = {}
for y, topics in per_year.items():
    cleaned[y] = normalize_year(int(y), topics)

with open(PR_TOPICS_FILE, "w", encoding="utf-8") as f:
    json.dump(cleaned, f, ensure_ascii=False, indent=2)